# 🎯 XERON — Colab fine-tuning (T4 1-GPU, parameter scaling)

Fine-tunes the [convaiinnovations/laya](https://huggingface.co/convaiinnovations/laya) multilingual checkpoint
on **XERON-ALL-v2 data (131,967 sequences: EN 6,000 + KR 82,344 + Browser 28,899 + WebAgent 14,724)**.

- Runtime: **GPU (T4 recommended; A100/T4 Pro also work)** — menu → Runtime → Change runtime type
- GRAD_ACCUM is scaled for 1 GPU to keep the same effective batch as 2xT4 (effective batch 64)
- Checkpoints are saved every epoch, so **you can resume even if the session drops**

> ⚠️ Free T4 sessions can be disconnected. Always save your results when done.


In [ ]:
# 1) GPU check
!nvidia-smi
import torch
print("CUDA:", torch.cuda.is_available(), "| GPUs:", torch.cuda.device_count())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print("GPU:", p.name, f"({p.total_memory/1e9:.1f} GB)")


In [ ]:
# 2) Install dependencies
!pip install -q "laya>=0.1.6" "transformers>=4.48.0" "datasets>=3.0.0" safetensors huggingface_hub pyarrow pandas scipy accelerate tabulate boto3 tqdm
import laya, transformers, torch
print("laya", laya.__version__, "| transformers", transformers.__version__, "| torch", torch.__version__)


In [ ]:
# 3) Clone the XERON repo (training/eval scripts)
!git clone --depth 1 https://github.com/PIXELZX0/XERON.git /content/XERON 2>/dev/null || (cd /content/XERON && git pull)
import sys; sys.path.insert(0, "/content/XERON")
%cd /content/XERON
!ls scripts/


## ⚙️ Settings — enter your values here

Editing the `SETTINGS` dictionary in the cell below propagates automatically to the rest of the pipeline.
- **Leaving a value empty** makes it read from Colab Secrets (🔑 icon → + New secret) under the same name
- ⚠️ **Putting S3 credentials directly in this cell works, but do not commit the notebook to GitHub**
  (this repo is public, so the keys would be exposed — using Colab Secrets is recommended)


In [ ]:
# ⚙️ Settings — enter values directly or read them from Secrets
SETTINGS = {
    # ── Data: S3-compatible (MinIO etc.) ───────────────────────
    # S3_KEY choices:
    #   train_items_all_multi4096_long.pt  (148,281 seq, includes long-context — recommended)
    #   train_items_all_multi4096.pt       (131,967 seq)
    #   train_items_long4096.pt            (16,314 seq — long-context only)
    "S3_ENDPOINT": "https://s3.flyingcart.kr",  # example; empty = Secrets
    "S3_BUCKET": "xeron",
    "S3_KEY": "train_items_all_multi4096_long.pt",
    "AWS_ACCESS_KEY_ID": "",      # empty = Secrets
    "AWS_SECRET_ACCESS_KEY": "",  # empty = Secrets

    # ── Base model ─────────────────────────────────────────────
    "BASE_MODEL": "multilingual", # ✅ multilingual (recommended; Korean/English/web all) | english(421M)

    # ── Training hyperparameters (parameter scaling) ───────────
    # Per-runtime profiles (MAX_LEN 4096 + effective batch 64):
    #   T4   (15GB): MICRO_BATCH=2,  GRAD_ACCUM=32, DTYPE=fp16
    #   A100 (40GB): MICRO_BATCH=8,  GRAD_ACCUM=8,  DTYPE=bf16  ← recommended
    #   A100 (80GB): MICRO_BATCH=16, GRAD_ACCUM=4,  DTYPE=bf16
    "EPOCHS": "2",                # 1~3 recommended (148K scale)
    "MICRO_BATCH": "8",           # adjust to GPU memory (base: A100 40GB)
    "GRAD_ACCUM": "8",            # effective batch = MICRO_BATCH × #GPUs × GRAD_ACCUM = 64
    "GROUP_SIZE": "4",
    "LR_ENCODER": "2.5e-5",
    "LR_HEAD": "1e-4",
    "DTYPE": "fp16",             # fp16(T4/V100) | bf16(A100/H100 recommended)
    "CHECKPOINT_EVERY": "1",      # checkpoint every N epochs (guards against session drops)
    "RESUME": "",                 # set to "auto" to resume

    # ── Context extension (MAX_LEN) ────────────────────────────
    # the multilingual base is RoPE-based (no position embeddings) — the default limit is
    # raised automatically by the scripts up to 32768 (4096x8) (CTX_CAP)
    # preprocess/train must use the same value! length ↑ = memory ↑ → MICRO_BATCH ↓
    #   profiles: 2048 (B recommended, MICRO_BATCH=4) / 4096 (long-context, MICRO_BATCH=2, MAX_TOKENS_BATCH=8192)
    "MAX_LEN": "2048",            # actual training context (choose between 1024~32768)
    "HEAD_MAX_LEN": "256",        # decision-head marker window
    "MAX_TOKENS_BATCH": "4096",   # token ceiling per micro-batch (8192 for the 4096 profile)

    # ── Results ────────────────────────────────────────────────
    "OUTPUT_NAME": "xeron-all-v2",
}

# Secrets fallback helper
try:
    from google.colab import userdata
    def _sec(name, default=""):
        try:
            return userdata.get(name, default) or default
        except Exception:
            return default
except ImportError:
    userdata = None
    def _sec(name, default=""):
        return default

for k in ["S3_ENDPOINT", "S3_BUCKET", "S3_KEY", "AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY"]:
    if not SETTINGS.get(k):
        SETTINGS[k] = _sec(k, SETTINGS.get(k, ""))

import os
os.environ.update({k: str(v) for k, v in SETTINGS.items() if v})
print("✅ Settings applied")
print("   S3:", SETTINGS["S3_ENDPOINT"] or "(not configured — Drive mode)", "| bucket:", SETTINGS["S3_BUCKET"])
print("   BASE_MODEL:", SETTINGS["BASE_MODEL"], "| EPOCHS:", SETTINGS["EPOCHS"],
      "| GRAD_ACCUM:", SETTINGS["GRAD_ACCUM"])


## 4) Prepare training data — 3 paths (automatic priority)

**① S3-compatible (MinIO etc.)** → **② Google Drive** → **③ Rebuild from JSONL**

- If S3 settings are provided or present in Secrets, it downloads from S3 automatically
- If S3 is not configured, it tries to mount Drive
- If neither is available, the last cell rebuilds directly from JSONL


In [ ]:
# 4A) Obtain training data (auto-try S3 → Drive → rebuild)
import os
ITEMS = "/content/train_items_all_v2.pt"

# ① S3-compatible (MinIO etc.)
if not os.path.exists(ITEMS) and SETTINGS.get("S3_ENDPOINT"):
    import boto3
    ep = SETTINGS["S3_ENDPOINT"]
    ck = dict(
        aws_access_key_id=SETTINGS["AWS_ACCESS_KEY_ID"],
        aws_secret_access_key=SETTINGS["AWS_SECRET_ACCESS_KEY"],
    )
    if "amazonaws.com" not in ep:  # S3-compatible such as MinIO/R2/B2
        ck["endpoint_url"] = ep
        ck["region_name"] = "us-east-1"
    s3 = boto3.client("s3", **{k: v for k, v in ck.items() if v})
    try:
        s3.download_file(SETTINGS["S3_BUCKET"], SETTINGS["S3_KEY"], ITEMS)
        print("☁️ Downloaded from S3:", ITEMS)
    except Exception as e:
        print("⚠️ S3 download failed:", str(e)[:200], "→ switching to Drive mode")

# ② Google Drive
if not os.path.exists(ITEMS):
    from google.colab import drive
    drive.mount("/content/drive")
    GDRIVE = "/content/drive/MyDrive/xeron/" + SETTINGS["S3_KEY"]
    if os.path.exists(GDRIVE):
        !cp "$GDRIVE" "$ITEMS"
        print("📁 Copied from Drive")

print("ITEMS exists:", os.path.exists(ITEMS),
      "| size:", (os.path.getsize(ITEMS) // 1048576) if os.path.exists(ITEMS) else 0, "MB")


In [ ]:
# 4B) Path ③ only: rebuild items from JSONL/HF (runs only when items is missing)
import os, torch, subprocess

def run_preprocess(model_dir, out_path, data_files=None):
    cmd = ["python", "scripts/preprocess.py",
           "--model-id", model_dir, "--output", out_path]
    if data_files:
        cmd += ["--data-files", data_files]
    else:
        cmd += ["--dataset", "LocalLLaMA/typed-decisions",
                "--config-name", "all", "--split", "train"]
    subprocess.run(cmd, check=True, cwd="/content/XERON")
    return torch.load(out_path, weights_only=False)

if not os.path.exists(ITEMS):
    GDRIVE = "/content/drive/MyDrive/xeron"
    KR_JSONL  = os.path.join(GDRIVE, "korean_typed.jsonl")
    BR_JSONL  = os.path.join(GDRIVE, "browser_typed.jsonl")
    M2W_JSONL = os.path.join(GDRIVE, "mind2web_typed.jsonl")
    os.makedirs("/content/items", exist_ok=True)
    print("no items → rebuilding from JSONL/HF (takes a few minutes)")
    en = run_preprocess(BASE_MODEL, "/content/items/en.pt")
    kr = run_preprocess(BASE_MODEL, "/content/items/kr.pt", KR_JSONL) if os.path.exists(KR_JSONL) else []
    br = run_preprocess(BASE_MODEL, "/content/items/br.pt", BR_JSONL) if os.path.exists(BR_JSONL) else []
    mw = run_preprocess(BASE_MODEL, "/content/items/mw.pt", M2W_JSONL) if os.path.exists(M2W_JSONL) else []
    all_items = en + kr + br + mw
    torch.save(all_items, ITEMS)
    print(f"Rebuild complete: {len(all_items)} sequences -> {ITEMS}")
else:
    print("items present — no rebuild needed:", ITEMS)


## 5) Download the base model

Downloads according to `SETTINGS["BASE_MODEL"]`:
- `multilingual` → `multilingual/` subfolder (mmBERT-base, 322M, includes Korean, recommended)
- `english` → repo root (ModernBERT-large, 421M)


In [ ]:
# 5) Download the base model
import os
from huggingface_hub import snapshot_download
from laya.agent import _fix_tokenizer_config

MODEL_ROOT = "/content/laya-model"
if SETTINGS["BASE_MODEL"] == "english":
    pattern = ["*.safetensors", "encoder/*", "tokenizer/*", "rl_agent_config.json"]
else:
    pattern = ["multilingual/*"]

if not os.path.exists(os.path.join(MODEL_ROOT, SETTINGS["BASE_MODEL"] == "english" and "model.safetensors" or "multilingual", "model.safetensors")):
    snapshot_download("convaiinnovations/laya", local_dir=MODEL_ROOT, allow_patterns=pattern)

BASE_MODEL = MODEL_ROOT if SETTINGS["BASE_MODEL"] == "english" else os.path.join(MODEL_ROOT, "multilingual")
_fix_tokenizer_config(os.path.join(BASE_MODEL, "tokenizer"))
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(os.path.join(BASE_MODEL, "tokenizer"))
print("Base model ready:", BASE_MODEL)
print("KR smoke test:", tok.encode("안녕하세요, 주문한 상품이 아직 안 왔어요.")[:8])


## 6) Hyperparameters — parameter scaling 🚀

Based on 1 GPU (T4). **Scales the same effective batch as 2xT4 (effective batch 64) to 1 GPU**:
- `GRAD_ACCUM=8` (scaled from 4 on 2 GPUs to 8) → effective batch = 8 × 1 × 8 = **64**
- `EPOCHS=2` (131K scale, estimated ~1–1.5 h/epoch on a single T4)
- `CHECKPOINT_EVERY=1` → checkpoint every epoch (guards against runtime drops)

| Experiment | EPOCHS | GRAD_ACCUM | Effective batch | Notes |
|---|---|---|---|---|
| A (quick validation) | 1 | 4 | 32 | ~30–45 min |
| **B (recommended)** | **2** | **8** | **64** | ~1.5–3 h |
| C (thorough) | 3 | 8 | 64 | Pro recommended |

> Change the values in the ⚙️ settings cell above (here they are passed on as env vars).


In [ ]:
# 6) Pass ⚙️ settings values on as training environment variables
import os
os.environ["EPOCHS"]           = SETTINGS["EPOCHS"]
os.environ["MICRO_BATCH"]      = SETTINGS["MICRO_BATCH"]
os.environ["GRAD_ACCUM"]       = SETTINGS["GRAD_ACCUM"]
os.environ["GROUP_SIZE"]       = SETTINGS["GROUP_SIZE"]
os.environ["LR_ENCODER"]       = SETTINGS["LR_ENCODER"]
os.environ["LR_HEAD"]          = SETTINGS["LR_HEAD"]
os.environ["CHECKPOINT_EVERY"] = SETTINGS["CHECKPOINT_EVERY"]
os.environ["RESUME"]           = SETTINGS["RESUME"]
os.environ["DTYPE"]            = SETTINGS.get("DTYPE", "fp16")
os.environ["MAX_LEN"]          = SETTINGS["MAX_LEN"]
os.environ["HEAD_MAX_LEN"]     = SETTINGS["HEAD_MAX_LEN"]
os.environ["MAX_TOKENS_BATCH"] = SETTINGS.get("MAX_TOKENS_BATCH", "4096")

OUT_DIR = f"/content/output/{SETTINGS['OUTPUT_NAME']}"
print("Effective batch:", int(SETTINGS["MICRO_BATCH"]) * int(SETTINGS["GRAD_ACCUM"]),
      "| EPOCHS:", SETTINGS["EPOCHS"], "| MAX_LEN(context):", SETTINGS["MAX_LEN"],
      "| OUT:", OUT_DIR)


## 7) Run training

- First run: just run it
- **Live display**: during training a progress bar (epoch/overall progress %), current loss, reward and estimated time remaining (ETA) are shown in real time
  - At the end of each epoch: elapsed time + remaining total time summary
  - The progress bar uses `\r`, so output does not grow and updates on a single line
- **If the runtime dropped**: set `RESUME` to `"auto"` in the ⚙️ settings cell and re-run the cells above
- **Logs preserved even if you close the session**: check the output under the Colab left tab → session state


In [ ]:
# 7) Run DDP training (1 GPU) — live progress / ETA display
import subprocess, os, sys, time

env = dict(os.environ, PYTHONUNBUFFERED="1")  # remove Python output buffering → live display
cmd = [
    "torchrun", "--standalone", "--nproc_per_node=1", "--max_restarts=0",
    "scripts/train_ddp.py", BASE_MODEL, OUT_DIR, ITEMS,
]
print("▶ Run:", " ".join(cmd), flush=True)
print("▶ Training started — progress (%/bar), loss and estimated time remaining (ETA) are shown live", flush=True)

t_start = time.time()
proc = subprocess.Popen(cmd, env=env, cwd="/content/XERON",
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
# stream character by character → the tqdm progress bar (\r updates) also appears live
while True:
    ch = proc.stdout.read(1)
    if not ch:
        break
    sys.stdout.write(ch.decode("utf-8", "replace"))
    sys.stdout.flush()
proc.wait()
if proc.returncode != 0:
    raise RuntimeError(f"💥 Training failed (exit code {proc.returncode}) — check the logs")

print(f"\n✅ Training complete: {OUT_DIR} (took {(time.time()-t_start)/60:.1f} min in total)")


## 8) Evaluation + save results

In [ ]:
# 8A) Benchmark evaluation
%cd /content/XERON
!python scripts/evaluate.py --model "$OUT_DIR" --device cuda --output /content/eval_results.json
import json
try:
    print(json.dumps(json.load(open("/content/eval_results.json"))["summary"], indent=2))
except Exception as e:
    print("Failed to parse evaluation results:", e)


In [ ]:
# 8B) Save results — S3 (if configured) or Drive
import os
if SETTINGS.get("S3_ENDPOINT"):
    import boto3
    ck = dict(aws_access_key_id=SETTINGS["AWS_ACCESS_KEY_ID"],
              aws_secret_access_key=SETTINGS["AWS_SECRET_ACCESS_KEY"])
    if "amazonaws.com" not in SETTINGS["S3_ENDPOINT"]:
        ck["endpoint_url"] = SETTINGS["S3_ENDPOINT"]
        ck["region_name"] = "us-east-1"
    s3 = boto3.client("s3", **{k: v for k, v in ck.items() if v})
    # upload the whole model directory
    for root, _dirs, files in os.walk(OUT_DIR):
        for f in files:
            local = os.path.join(root, f)
            key = f"output/{os.path.basename(OUT_DIR)}/{os.path.relpath(local, OUT_DIR)}"
            s3.upload_file(local, SETTINGS["S3_BUCKET"], key)
    s3.upload_file("/content/eval_results.json", SETTINGS["S3_BUCKET"], "output/eval_results.json")
    print("☁️ Uploaded results to S3")
else:
    !mkdir -p "/content/drive/MyDrive/xeron/output"
    !cp -r "$OUT_DIR" "/content/drive/MyDrive/xeron/output/"
    !cp /content/eval_results.json "/content/drive/MyDrive/xeron/output/"
    print("📁 Copied results to Drive")


In [ ]:
# 9) [optional] Upload to HuggingFace Hub (requires HF_TOKEN)
# !HF_TOKEN=hf_xxx python scripts/upload_hf.py --model-dir "$OUT_DIR" --repo-id PIXELZX/XERON
print("skip or uncomment with your HF_TOKEN")


## 📋 Quick checklist

1. Runtime → Change runtime type → **T4 GPU** ✅
2. Check values in the **⚙️ settings cell** (if S3 is not configured, upload `xeron/train_items_all_v2.pt` to Drive) ✅
3. Run cells 1~6 in order ✅
4. Run training (cell 7) — a checkpoint is saved automatically every epoch ✅
5. Evaluate + save results (cell 8) — automatic to S3/Drive ✅
6. If it drops, resume with `RESUME="auto"` ✅

### Notes when using S3 (MinIO)
- Register in the notebook's left 🔑 **Secrets**: `S3_ENDPOINT`, `S3_BUCKET`, `S3_KEY`,
  `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY` (putting them directly in the settings cell also works)
- MinIO must be reachable from Colab's external network (public IP / port forwarding / tunnel)
